<a href="https://colab.research.google.com/github/ggalarza1/accounting/blob/main/Bitcoin_Wallet_Accounting_system.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#!/usr/bin/env python3
"""
Bitcoin Wallet Accounting System - Multi-xpub + Financial Statements
Generates professional Income Statement, Balance Sheet, and Cash Flow
Supports multiple xpubs for comprehensive business accounting
"""

import json
from datetime import datetime
from typing import List, Dict
from collections import defaultdict


def generate_income_statement(transactions: List[Dict], start_date: str = None, end_date: str = None) -> Dict:
    """
    Generate Income Statement (Profit & Loss) for Bitcoin operations
    Similar to traditional P&L but adapted for crypto:
    - Revenue: Bitcoin received as payment/mining/staking
    - Expenses: Fees paid, transaction costs
    - Gains/Losses: Realized from sales/dispositions
    """
    # Filter by date range if provided
    filtered_txs = transactions
    if start_date:
        filtered_txs = [t for t in filtered_txs if t.get('date', '') >= start_date]
    if end_date:
        filtered_txs = [t for t in filtered_txs if t.get('date', '') <= end_date]

    # Categorize transactions
    revenue_items = defaultdict(float)
    expense_items = defaultdict(float)
    gain_loss_items = defaultdict(float)

    for tx in filtered_txs:
        tx_type = tx.get('type', 'unknown')
        usd_value = tx.get('net_change_usd', 0) or 0
        realized_gain = tx.get('realized_gain_loss', 0) or 0

        if tx_type == 'incoming':
            revenue_items['Bitcoin Revenue'] += abs(usd_value)

        elif tx_type == 'outgoing':
            fee_usd = tx.get('usd_fee', 0) or 0
            if fee_usd > 0:
                expense_items['Transaction Fees'] += fee_usd

            if realized_gain != 0:
                if realized_gain > 0:
                    gain_loss_items['Realized Gains'] += realized_gain
                else:
                    gain_loss_items['Realized Losses'] += abs(realized_gain)

    total_revenue = sum(revenue_items.values())
    total_expenses = sum(expense_items.values())
    net_gains = sum(v for k, v in gain_loss_items.items() if 'Gains' in k)
    net_losses = sum(v for k, v in gain_loss_items.items() if 'Losses' in k)

    net_income = total_revenue - total_expenses + net_gains - net_losses

    return {
        'statement_type': 'Income Statement',
        'period': f"{start_date or 'All Time'} to {end_date or 'Present'}",
        'generated_at': datetime.now().isoformat(),
        'revenue': {'items': dict(revenue_items), 'total': total_revenue},
        'expenses': {'items': dict(expense_items), 'total': total_expenses},
        'gains_losses': {
            'realized_gains': net_gains,
            'realized_losses': net_losses,
            'net': net_gains - net_losses
        },
        'net_income': net_income,
        'summary': {
            'total_transactions': len(filtered_txs),
            'period_start': start_date or 'Beginning',
            'period_end': end_date or 'Present'
        }
    }


def generate_balance_sheet(summary: Dict, as_of_date: str = None) -> Dict:
    """
    Generate Balance Sheet - Assets, Liabilities, Equity
    For a Bitcoin wallet: Assets = Bitcoin held at fair market value
    """
    current_btc = summary.get('current_balance_btc', 0)
    current_price = summary.get('current_btc_price_usd', 0)
    unrealized = summary.get('unrealized_gains_losses_usd', 0)

    btc_value_usd = current_btc * current_price
    cost_basis = btc_value_usd - unrealized if btc_value_usd > 0 else 0
    total_invested = summary.get('total_received_usd', 0)
    total_realized_gains = summary.get('realized_gains_losses_usd', 0)

    return {
        'statement_type': 'Balance Sheet',
        'as_of_date': as_of_date or datetime.now().strftime('%Y-%m-%d'),
        'generated_at': datetime.now().isoformat(),
        'assets': {
            'current_assets': {
                'bitcoin_holding_btc': current_btc,
                'bitcoin_fair_value_usd': btc_value_usd,
                'cost_basis': cost_basis,
                'unrealized_gain_loss': unrealized
            },
            'total_assets': btc_value_usd
        },
        'liabilities': {
            'notes': 'No recorded liabilities for self-custody wallet',
            'total_liabilities': 0
        },
        'equity': {
            'contributed_capital': total_invested,
            'retained_earnings': total_realized_gains,
            'unrealized_gains': unrealized,
            'total_equity': btc_value_usd
        },
        'summary': {
            'btc_price_used': current_price
        }
    }


def generate_comprehensive_report(wallet_data: Dict, start_date: str = None, end_date: str = None) -> Dict:
    """Generate all financial statements in one report"""
    transactions = wallet_data.get('transactions', [])

    return {
        'report_metadata': {
            'generated_at': datetime.now().isoformat(),
            'report_period': f"{start_date or 'All Time'} to {end_date or 'Present'}",
            'wallet_type': 'xpub' if wallet_data.get('xpub') else 'single_address'
        },
        'income_statement': generate_income_statement(transactions, start_date, end_date),
        'balance_sheet': generate_balance_sheet(wallet_data, end_date),
        'summary_metrics': {
            'total_realized_gains': wallet_data.get('realized_gains_losses_usd', 0),
            'total_unrealized_gains': wallet_data.get('unrealized_gains_losses_usd', 0),
            'current_btc_holdings': wallet_data.get('current_balance_btc', 0),
            'current_btc_value_usd': wallet_data.get('current_balance_btc', 0) * wallet_data.get('current_btc_price_usd', 0)
        }
    }


def print_financial_report(report: Dict):
    """Pretty print the financial statements"""
    meta = report['report_metadata']
    print("\n" + "=" * 80)
    print("📊 BITCOIN WALLET FINANCIAL STATEMENTS")
    print("=" * 80)
    print(f"Period: {meta['report_period']}")
    print(f"Generated: {meta['generated_at'][:19]}")

    # Income Statement
    inc = report['income_statement']
    print("\n" + "-" * 80)
    print("📈 INCOME STATEMENT")
    print("-" * 80)
    print(f"\n{'Total Revenue':.<50} ${inc['revenue']['total']:>15,.2f}")
    print(f"{'Total Expenses':.<50} ${inc['expenses']['total']:>15,.2f}")
    print(f"{'Net Gains/Losses':.<50} ${inc['gains_losses']['net']:>15,.2f}")
    print(f"{'NET INCOME':.<50} ${inc['net_income']:>15,.2f}")

    # Balance Sheet
    bs = report['balance_sheet']
    print("\n" + "-" * 80)
    print("📊 BALANCE SHEET")
    print("-" * 80)
    ca = bs['assets']['current_assets']
    print(f"\nBitcoin Holdings: {ca['bitcoin_holding_btc']:.8f} BTC")
    print(f"{'Fair Market Value':.<50} ${ca['bitcoin_fair_value_usd']:>15,.2f}")
    print(f"{'Cost Basis':.<50} ${ca['cost_basis']:>15,.2f}")
    print(f"{'Unrealized G/L':.<50} ${ca['unrealized_gain_loss']:>15,.2f}")
    print(f"{'TOTAL ASSETS/EQUITY':.<50} ${bs['assets']['total_assets']:>15,.2f}")

    print("\n" + "=" * 80)


if __name__ == "__main__":
    # Example usage
    print("Bitcoin Accounting Statements Generator")
    print("This module provides professional financial reporting")
    print("\nUse with wallet data from bitcoin_wallet_extractor.py")
    print("Example: Load wallet_data.json, pass to generate_comprehensive_report()")

Bitcoin Accounting Statements Generator
This module provides professional financial reporting

Use with wallet data from bitcoin_wallet_extractor.py
Example: Load wallet_data.json, pass to generate_comprehensive_report()
